# Notebook 04c — Species Weight Grid Search

**Goal:** Find the optimal species boost weight for the anti_pollution strategy.

**Why this matters:**
- Species boost is the primary mechanism (72% of MAP gain from ablation)
- Default weight=0.5 was chosen without grid search
- No reindex needed — just change the query weight and re-retrieve

**What we found from v2→v3 rebuild:**
- Species detection was broken (87% unknown via free-text)
- Fixed using MeSH controlled vocabulary headings
- MAP went from 0.151 → 0.180 just from fixing the data quality bug

**This notebook:** Now that species detection is correct, find the optimal weight.


In [ ]:
import os, re, json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from elasticsearch import Elasticsearch
from ranx import Qrels, Run, evaluate

es = Elasticsearch(host='elasticsearch')
print('ES:', es.info()['version']['number'])

INDEX        = 'medline_enriched'
QRELS_FILE   = '/work/data/FIR-s05-training-qrels.txt'
QUERIES_FILE = '/work/data/FIR-s05-training-queries-simple.txt'
RUNS_DIR     = '/work/runs'

def load_queries(path):
    q = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                q[parts[0]] = parts[1].strip('"')
    return q

def load_ranx_qrels(path):
    data = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4 and int(parts[3]) > 0:
                data.setdefault(parts[0], {})[parts[2]] = int(parts[3])
    return Qrels(data)

def load_ranx_run(path):
    data = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                data.setdefault(parts[0], {})[parts[2]] = float(parts[4])
    return Run(data)

SPECIES_PATTERNS = [
    (re.compile(r'\bhomo\s+sapiens\b|\bhuman\b', re.I), 'Homo sapiens'),
    (re.compile(r'\bmus\s+musculus\b|\bmouse\b',  re.I), 'Mus musculus'),
    (re.compile(r'\brattus\b|\brat\b',            re.I), 'Rattus norvegicus'),
    (re.compile(r'\bdrosophila\b',                  re.I), 'Drosophila melanogaster'),
]

def get_species(t):
    for p, n in SPECIES_PATTERNS:
        if p.search(t): return n
    return None

queries = load_queries(QUERIES_FILE)
qrels   = load_ranx_qrels(QRELS_FILE)

BASELINE_MAP = 0.0855
V2_MAP       = 0.1508   # species detection broken (87% unknown)
V3_MAP       = 0.1800   # species detection fixed (MeSH-first)

print(f'Queries: {len(queries)}')
print(f'BM25 baseline:      MAP {BASELINE_MAP}')
print(f'Anti-pollution v2:  MAP {V2_MAP}  (species 87% unknown)')
print(f'Anti-pollution v3:  MAP {V3_MAP}  (species fixed, weight=0.5, k1=1.5 b=0.30)')
print()
print('Goal: find optimal species weight now that detection is correct')


---
## 1. Verify species detection is fixed

In [ ]:
# Confirm species distribution before running grid
resp = es.search(index=INDEX, body={
    'aggs': {'species': {'terms': {'field': 'species_norm', 'size': 10}}},
    'size': 0
})
print('species_norm distribution (should be ~15% unknown after fix):')
total = sum(b['doc_count'] for b in resp['aggregations']['species']['buckets'])
for b in resp['aggregations']['species']['buckets']:
    pct = b['doc_count'] / 263080 * 100
    bar = '█' * int(pct / 2)
    print(f"  {b['key']:<30} {b['doc_count']:>8,}  ({pct:>5.1f}%)  {bar}")

unknown_pct = (263080 - total) / 263080 * 100
print(f'  {"unknown":<30} {263080-total:>8,}  ({unknown_pct:>5.1f}%)')
print()
if unknown_pct < 20:
    print('✓ Species detection fixed — MeSH-first approach working')
    print('  Proceed with weight grid search')
else:
    print('✗ Still too many unknown — check 07_enrich_index.py was rebuilt')
    print('  Run: docker exec fir-workbench python scripts/07_enrich_index.py --recreate')


---
## 2. Species weight grid search

In [ ]:
def retrieve_weighted(qid, qtext, species_weight, size=1000):
    species = get_species(qtext)
    base = {
        'bool': {
            'should': [
                {'multi_match': {'query': qtext,
                                 'fields': ['TI^2','AB','mesh_clean^1.5'],
                                 'type': 'best_fields'}},
                {'match': {'gene_context':    {'query': qtext, 'boost': 3.0}}},
                {'match': {'concept_context': {'query': qtext, 'boost': 1.5}}},
            ],
            'minimum_should_match': 1,
        }
    }
    funcs = []
    if species:
        funcs.append({'filter': {'term': {'species_norm': species}},
                      'weight': species_weight})
    funcs.append({'gauss': {'pub_year': {'origin': 2002, 'scale': 3, 'decay': 0.5}},
                  'weight': 0.05})
    resp = es.search(index=INDEX, body={
        'query': {'function_score': {'query': base, 'functions': funcs,
                                     'score_mode': 'sum', 'boost_mode': 'multiply'}},
        'size': size, '_source': ['PMID']
    }, request_timeout=30)
    return resp['hits']['hits']


def run_weight_experiment(weight):
    tag      = f'tune_w{str(weight).replace(".","_")}'
    run_path = f'{RUNS_DIR}/{tag}.run'
    with open(run_path, 'w') as f:
        for qid, qtext in queries.items():
            for rank, hit in enumerate(retrieve_weighted(qid, qtext, weight), 1):
                pmid = hit.get('_source', {}).get('PMID', hit['_id'])
                f.write(f'{qid} Q0 {pmid} {rank} {hit["_score"]:.6f} {tag}\n')
    run = load_ranx_run(run_path)
    return evaluate(qrels, run, ['map','recall@100','precision@5']), run_path


print('Species weight grid search on FIXED species detection...')
print('(9 weights × 38 queries = ~5 min)\n')

weights = [0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0]
weight_results = {}

print(f'{"Weight":>8} {"Mult":>6}  {"MAP":>8} {"R@100":>8} {"P@5":>8} {"vs v3":>8}')
print('─' * 62)
for w in weights:
    r, _ = run_weight_experiment(w)
    weight_results[w] = r
    delta = (r['map'] - V3_MAP) / V3_MAP * 100
    marker = ' ← current' if w == 0.5 else ''
    print(f'{w:>8} {1+w:>5.1f}x  {r["map"]:>8.4f} {r["recall@100"]:>8.4f} '
          f'{r["precision@5"]:>8.4f} {delta:>+7.1f}%{marker}')

best_w   = max(weight_results, key=lambda w: weight_results[w]['map'])
best_map = weight_results[best_w]['map']
print(f'\n✓ Best weight: {best_w}  MAP={best_map:.4f}  '
      f'({(best_map-V3_MAP)/V3_MAP*100:+.1f}% vs v3)')


In [ ]:
# Visualise
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric, label in [
    (axes[0], 'map',         'MAP'),
    (axes[1], 'recall@100',  'recall@100'),
    (axes[2], 'precision@5', 'P@5'),
]:
    vals = [weight_results[w][metric] for w in weights]
    ax.plot(weights, vals, 'o-', color='#2E75B6', linewidth=2, markersize=7)
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='v3 (0.5)')
    ax.axvline(best_w, color='#E74C3C', linestyle=':', alpha=0.8,
               label=f'best ({best_w})')
    ax.set_xlabel('Species weight')
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs species weight')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Species Weight Grid Search (MeSH-fixed index)\n'
             'function_score: 1.0 base + weight = multiplier', fontsize=11)
plt.tight_layout()
plt.savefig(f'{RUNS_DIR}/species_weight_grid.png', dpi=120)
plt.show()

print(f'Optimal multiplier: {1+best_w:.1f}x (weight={best_w})')
print(f'Save best run as final: {RUNS_DIR}/tune_w{str(best_w).replace(".","_")}.run')


---
## 3. Summary

In [ ]:
print('='*60)
print('NOTEBOOK 04c SUMMARY')
print('='*60)
print()
print('Species detection fix (data quality):')
print(f'  Before: 87% unknown  → species boost at 13% capacity')
print(f'  After:  ~15% unknown → species boost at full capacity')
print()
print('MAP progression:')
print(f'  BM25 baseline:      {BASELINE_MAP:.4f}')
print(f'  Anti-pollution v2:  {V2_MAP:.4f}  (species broken)')
print(f'  Anti-pollution v3:  {V3_MAP:.4f}  (species fixed, weight=0.5)')
if weight_results:
    print(f'  Best weight ({best_w}):   {best_map:.4f}  (tuned weight)')
    improvement = (best_map - BASELINE_MAP) / BASELINE_MAP * 100
    print()
    print(f'  Total improvement over baseline: +{improvement:.0f}%')
print()
print(f'Next: Notebook 04 (renamed from 05) — full evaluation')
print(f'  Use run file: tune_w{str(best_w).replace(".","_")}.run as final system')
